# 딥러닝심화 과제 4 - GPT2 기반 챗봇 Gradio 인터페이스 구현
#### 컴퓨터공학부 인공지능공학과 20231049 정우제
<hr>

## 목차
<hr>

0. [필요한 라이브러리 불러오기 및 환경설정](#0-필요한-라이브러리-불러오기-및-환경설정)
1. [학습된 모델 로드](#1-학습된-모델-로드)
2. [챗봇 응답 생성 함수 구현](#2-챗봇-응답-생성-함수-구현)
3. [Gradio 인터페이스 구축 및 실행](#3-gradio-인터페이스-구축-및-실행)

## 0. 필요한 라이브러리 불러오기 및 환경설정
<hr>

In [28]:
import gradio as gr
import numpy as np
import torch
import random
import time
from transformers import GPT2LMHeadModel, AutoTokenizer

In [29]:
is_cuda = torch.cuda.is_available()
is_mps = torch.backends.mps.is_available()
device = torch.device('cuda' if is_cuda else 'mps' if is_mps else 'cpu')
print(device)

cuda


## 1. 학습된 모델 로드
<hr>

In [30]:
# 모델들 로드
models = {}
tokenizers = {}

model_paths = {
    'model_all': './model_all',
    'model_brother': './model_brother',
    'model_friend': './model_friend',
    'model_girlfriend': './model_girlfriend'
}

model_names = {
    'model_all': '전체 통합 모델',
    'model_brother': '동생 모델',
    'model_friend': '친구 모델',
    'model_girlfriend': '여자친구 모델'
}


In [40]:
for model_key, model_path in model_paths.items():
    models[model_key] = GPT2LMHeadModel.from_pretrained(model_path)
    tokenizers[model_key] = AutoTokenizer.from_pretrained(model_path)
    models[model_key].to(device)
    print(f"{model_names[model_key]} 로드 완료!")

전체 통합 모델 로드 완료!
동생 모델 로드 완료!
친구 모델 로드 완료!
여자친구 모델 로드 완료!


## 2. 챗봇 응답 생성 함수 구현
<hr>

* 사용자 입력에 대해 선택된 모델이 한 번의 응답을 생성함.

In [34]:
def return_answer_by_chatbot(user_text, model_type='model_all'):
    model = models[model_type]
    tokenizer = tokenizers[model_type]
    
    sent = '<usr>' + user_text + '<sys>'
    input_ids = [tokenizer.bos_token_id] + tokenizer.encode(sent)
    input_ids = torch.tensor([input_ids], dtype=torch.long).to(device)
    
    output = model.generate(
        input_ids, 
        max_length=128,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id,
        early_stopping=True
    )
    
    sentence = tokenizer.decode(output[0].tolist())
    chatbot_response = sentence.split('<sys>')[1].replace('</s>', '').strip()
    return chatbot_response

* 초기 입력 메시지로부터 시작하여 설정한 turn수에 맞게 자문자답으로 대화를 생성함.
* 챗봇이 이전응답을 다음 입력으로 사용함.

In [82]:
def multi_turn_chat(initial_message, num_turns, model_type='model_all'):
    """Multi-turn 연속 대화 생성"""
    conversation = []
    
    conversation.append({'role': 'user', 'content': initial_message})
    
    current_input = initial_message
    
    for turn in range(num_turns - 1):  
        # 봇이 현재 입력에 대한 응답 생성
        bot_response = return_answer_by_chatbot(current_input, model_type)
        
        # 짝수 턴: 왼쪽, 홀수 턴: 오른쪽
        if turn % 2 == 0:  # 0, 2, 4, ... (왼쪽)
            conversation.append({'role': 'assistant', 'content': bot_response})
        else:  # 1, 3, 5, ... (오른쪽)
            conversation.append({'role': 'user', 'content': bot_response})
        
        current_input = bot_response
        
    return conversation

In [83]:
def chat_interface(message, num_turns, model_type):
    if not message.strip():
        return []
    
    # Multi-turn 대화 생성
    history = multi_turn_chat(message, num_turns, model_type)
    
    return history

## 3. Gradio 인터페이스 구축 및 실행
<hr>

In [86]:
# Gradio 인터페이스 구성
with gr.Blocks(title="HW04_20231049_정우제 챗봇", css="""
    .chatbot {background-color: #B2C7D9;}
    """) as demo:
    
    gr.Markdown("# 💬 HW04_20231049_정우제 챗봇")
    
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="대화",
                height=600,
                type='messages'
            )
            
        with gr.Column(scale=1):
            model_selector = gr.Radio(
                choices=[
                    ('전체', 'model_all'),
                    ('동생', 'model_brother'),
                    ('친구', 'model_friend'),
                    ('여친', 'model_girlfriend')
                ],
                value='model_all',
                label="모델"
            )
            
            num_turns = gr.Slider(
                minimum=2,
                maximum=20,
                value=10,
                step=1,
                label="대화 턴 수"
            )
            
            msg = gr.Textbox(
                label="메시지",
                placeholder="메시지를 입력하세요...",
                lines=2
            )
            
            with gr.Row():
                submit_btn = gr.Button("전송", variant="primary", size="lg")
                clear_btn = gr.Button("초기화", size="lg")
    
    submit_btn.click(
        fn=chat_interface,
        inputs=[msg, num_turns, model_selector],
        outputs=[chatbot]
    )
    
    msg.submit(
        fn=chat_interface,
        inputs=[msg, num_turns, model_selector],
        outputs=[chatbot]
    )
    
    clear_btn.click(
        fn=lambda: [],
        outputs=[chatbot]
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7885
* Running on public URL: https://d8a477224edbd33249.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
